In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
X_train = pd.read_csv("../data/processed/X_train_smote.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train_smote.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [3]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_grid = {

    "n_estimators":[100,200,300],

    "max_depth":[3,5,7],

    "learning_rate":[0.01,0.05,0.1],

    "subsample":[0.8,1.0],

    "colsample_bytree":[0.8,1.0]
}

In [5]:
grid_search = GridSearchCV(

    estimator=xgb,

    param_grid=param_grid,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1,

    verbose=2
)

grid_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=True,
                                     eval_metric='logloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_p...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [3, 5, 7],
                         'n_estimators': [100, 200, 300],
                         'subsample': [0.8, 1.0]},
             scoring='roc_auc', verbose=2)

In [6]:
print(grid_search.best_params_)

{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}


In [7]:
print(grid_search.best_score_)

0.9999921267312347


In [8]:
lgbm = LGBMClassifier(
    random_state=42
)

param_dist = {

    "n_estimators":[100,200,300,400],

    "learning_rate":[0.01,0.05,0.1],

    "num_leaves":[31,50,70],

    "max_depth":[5,7,10],

    "subsample":[0.8,1.0]
}

In [9]:
random_search = RandomizedSearchCV(

    estimator=lgbm,

    param_distributions=param_dist,

    n_iter=20,

    scoring="roc_auc",

    cv=cv,

    random_state=42,

    n_jobs=-1,

    verbose=2
)

random_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[LightGBM] [Info] Number of positive: 226602, number of negative: 226602
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.089257 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7905
[LightGBM] [Info] Number of data points in the train set: 453204, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=LGBMClassifier(random_state=42), n_iter=20,
                   n_jobs=-1,
                   param_distributions={'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [5, 7, 10],
                                        'n_estimators': [100, 200, 300, 400],
                                        'num_leaves': [31, 50, 70],
                                        'subsample': [0.8, 1.0]},
                   random_state=42, scoring='roc_auc', verbose=2)

In [10]:
print(random_search.best_params_)

{'subsample': 0.8, 'num_leaves': 70, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.1}


In [11]:
best_xgb = grid_search.best_estimator_

pred = best_xgb.predict(X_test)

prob = best_xgb.predict_proba(X_test)[:,1]

print("ROC AUC:",roc_auc_score(y_test,prob))

print("Precision:",precision_score(y_test,pred))

print("Recall:",recall_score(y_test,pred))

print("F1:",f1_score(y_test,pred))

ROC AUC: 0.9726961664633597
Precision: 0.7755102040816326
Recall: 0.8
F1: 0.7875647668393783


In [12]:
best_lgbm = random_search.best_estimator_

pred = best_lgbm.predict(X_test)

prob = best_lgbm.predict_proba(X_test)[:,1]

print("ROC AUC:",roc_auc_score(y_test,prob))

print("Precision:",precision_score(y_test,pred))

print("Recall:",recall_score(y_test,pred))

print("F1:",f1_score(y_test,pred))

ROC AUC: 0.971483199534732
Precision: 0.8409090909090909
Recall: 0.7789473684210526
F1: 0.8087431693989071


In [13]:
scores = cross_val_score(

    best_xgb,

    X_train,

    y_train,

    cv=cv,

    scoring="roc_auc"
)

print(scores)

print(scores.mean())

[0.9999964  0.99999998 0.99999893 0.99998384 0.99998514]
0.999992859538475


In [14]:
scores = cross_val_score(

    best_lgbm,

    X_train,

    y_train,

    cv=cv,

    scoring="roc_auc"
)

print(scores.mean())

[LightGBM] [Info] Number of positive: 181282, number of negative: 181281
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073594 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7905
[LightGBM] [Info] Number of data points in the train set: 362563, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500001 -> initscore=0.000006
[LightGBM] [Info] Start training from score 0.000006
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [15]:
comparison = pd.DataFrame({

    "Model":[
        "Tuned XGBoost",
        "Tuned LightGBM"
    ],

    "ROC AUC":[
        roc_auc_score(
            y_test,
            best_xgb.predict_proba(X_test)[:,1]
        ),

        roc_auc_score(
            y_test,
            best_lgbm.predict_proba(X_test)[:,1]
        )
    ]
})

comparison

,Model,ROC AUC
0,Tuned XGBoost,0.972696
1,Tuned LightGBM,0.971483


In [16]:
comparison.to_csv(
    "../reports/tuning_results.csv",
    index=False
)

In [17]:
joblib.dump(

    best_xgb,

    "../models/tuned_xgboost.pkl"
)

joblib.dump(

    best_lgbm,

    "../models/tuned_lightgbm.pkl"
)

['../models/tuned_lightgbm.pkl']

In [18]:
if comparison.iloc[0]["ROC AUC"] > comparison.iloc[1]["ROC AUC"]:

    final_model = best_xgb

else:

    final_model = best_lgbm

In [19]:
joblib.dump(

    final_model,

    "../models/final_model.pkl"
)

['../models/final_model.pkl']

In [20]:
cv_results = pd.DataFrame({

    "Fold":[1,2,3,4,5],

    "ROC AUC":scores
})

cv_results.to_csv(

    "../reports/cross_validation_results.csv",

    index=False
)